In [ ]:
# Client Setup
import boto3

client = boto3.client("bedrock-runtime", region_name="us-west-2")
model_id = "us.anthropic.claude-sonnet-4-20250514-v1:0"


In [9]:
# Helper functions


def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {
            "role": "assistant",
            "content": [{"text": content}],
        }
    else:
        assistant_message = {"role": "assistant", "content": content}

    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice="auto",
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    tool_choices = {
        "auto": {"auto": {}},
        "any": {"any": {}},
    }
    if tools:
        choice = tool_choices.get(tool_choice, {"tool": {"name": tool_choice}})
        params["toolConfig"] = {"tools": tools, "toolChoice": choice}

    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p]),
    }

In [10]:
# Tool Schemas

article_details_schema = {
    "toolSpec": {
        "name": "article_details",
        "description": "This tool should be called with details about an article. It accepts information about the article's title, author, and related topics.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The title of the article. Can be left empty.",
                    },
                    "author": {
                        "type": "string",
                        "description": "The name of the article's author. Can be left empty.",
                    },
                    "topics": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "A list of topics or categories that the article covers. Can be an empty list.",
                    },
                },
            }
        },
    }
}

to_json_schema = {
    "toolSpec": {
        "name": "to_json",
        "description": "This tool processes any JSON data and can be used for generating structured content, transforming information, or creating any JSON-based output needed for your task.",
        "inputSchema": {
            "json": {"type": "object", "additionalProperties": True}
        },
    }
}

In [11]:
messages = []

add_user_message(
    messages,
    "Write a one-paragraph scholarly article about computer science. Include a title and author name",
)

result = chat(messages)

add_assistant_message(messages, result["text"])

result["text"]

'**Title: Quantum-Classical Hybrid Algorithms: Bridging Computational Paradigms for Near-Term Quantum Advantage**\n\n**Author: Dr. Sarah Chen, Department of Computer Science, Stanford University**\n\nThe emergence of noisy intermediate-scale quantum (NISQ) devices has necessitated the development of quantum-classical hybrid algorithms that leverage the complementary strengths of both computational paradigms while mitigating the limitations inherent in current quantum hardware. These hybrid approaches, exemplified by variational quantum algorithms such as the Quantum Approximate Optimization Algorithm (QAOA) and Variational Quantum Eigensolver (VQE), employ quantum processors to explore high-dimensional solution spaces through quantum superposition and entanglement, while classical optimization techniques iteratively refine parameter sets to minimize cost functions. Recent theoretical analysis demonstrates that such hybrid frameworks can achieve polynomial quantum speedups for specific 

In [12]:
messages = []

add_user_message(
    messages,
    f"""
Analyze the article below and extract key data. Then call the article_details tool.
                 
<article_text>
{result["text"]}                 
</article_text>
""",
)

json_result = chat(
    messages, tools=[article_details_schema], tool_choice="article_details"
)

json_result

{'parts': [{'toolUse': {'toolUseId': 'tooluse_r3G38zV8yRNGeyL0ZrrC5w',
    'name': 'article_details',
    'input': {'title': 'Quantum-Classical Hybrid Algorithms: Bridging Computational Paradigms for Near-Term Quantum Advantage',
     'author': 'Dr. Sarah Chen',
     'topics': ['quantum computing',
      'hybrid algorithms',
      'NISQ devices',
      'variational quantum algorithms',
      'QAOA',
      'VQE',
      'quantum optimization',
      'quantum chemistry',
      'computational paradigms',
      'quantum advantage']},
    'type': 'tool_use'}}],
 'stop_reason': 'tool_use',
 'text': ''}

In [13]:
messages = []

add_user_message(
    messages,
    f"""
Analyze the article below and extract key data. Then call the to_json tool.
                 
<article_text>
{result["text"]}                 
</article_text>

When you call to_json, pass in the following structure:
{{
    "title": str # title of the article,
    "author": str # author of the article,
    "topics": List[str] # List of topics mentioned in the article,
    "num_topics: int # Number of topics mentioned
}}
""",
)

flexible_result = chat(messages, tools=[to_json_schema], tool_choice="to_json")

In [14]:
flexible_result

{'parts': [{'toolUse': {'toolUseId': 'tooluse_EH4oyZ3o03oIS0qFvKFZPb',
    'name': 'to_json',
    'input': {'title': 'Quantum-Classical Hybrid Algorithms: Bridging Computational Paradigms for Near-Term Quantum Advantage',
     'author': 'Dr. Sarah Chen, Department of Computer Science, Stanford University',
     'topics': ['quantum-classical hybrid algorithms',
      'NISQ devices',
      'variational quantum algorithms',
      'Quantum Approximate Optimization Algorithm (QAOA)',
      'Variational Quantum Eigensolver (VQE)',
      'quantum superposition',
      'quantum entanglement',
      'classical optimization',
      'combinatorial optimization',
      'quantum chemistry',
      'quantum speedups',
      'gate errors',
      'coherence times',
      'fault-tolerant integration',
      'quantum advantage'],
     'num_topics': 15},
    'type': 'tool_use'}}],
 'stop_reason': 'tool_use',
 'text': ''}